# 📊 02 - Stock Data Ingestion
## Download Nifty 100 Stocks and Nifty Index Data from Yahoo Finance

---

**Notebook:** 02_stock_ingestion.ipynb  
**Project:** Financial Market Trend Forecasting  
**Data Source:** Yahoo Finance (yfinance)  
**Data Window:** 2023-01-01 to 2026-01-01

---

## Objective

Download and construct the raw stock price dataset for all Nifty 100 stocks and Nifty index.

## Output

- File: `Market_Data/raw/raw_stock_data.csv`
- Format: Long format with columns: Date, Ticker, Open, High, Low, Close, Volume

## 1. Setup and Imports

In [48]:
# Install required packages if not available
# !pip install yfinance pandas numpy tqdm

In [49]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
import os
import warnings
from tqdm import tqdm
import time

warnings.filterwarnings('ignore')

print(f"Pandas version: {pd.__version__}")
print(f"yfinance version: {yf.__version__}")

Pandas version: 2.3.3
yfinance version: 1.2.0


In [50]:
# Configuration
START_DATE = "2023-01-01"
END_DATE = "2026-01-01"

# Output paths
RAW_DATA_DIR = "../Market_Data/raw"
OUTPUT_FILE = os.path.join(RAW_DATA_DIR, "raw_stock_data.csv")

# Create directory if not exists
os.makedirs(RAW_DATA_DIR, exist_ok=True)

print(f"Data Window: {START_DATE} to {END_DATE}")
print(f"Output Path: {OUTPUT_FILE}")

Data Window: 2023-01-01 to 2026-01-01
Output Path: ../Market_Data/raw\raw_stock_data.csv


## 2. Nifty 100 Ticker List

We'll use a curated list of Nifty 100 tickers. The `.NS` suffix is required for Yahoo Finance to identify NSE (National Stock Exchange) stocks.

In [51]:
# Nifty 100 stock tickers (Nifty 50 + Nifty Next 50 = 100 stocks)
# These are the NSE symbols without the .NS suffix

NIFTY_100_TICKERS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK", "HINDUNILVR", "SBIN", "BHARTIARTL", "KOTAKBANK", "ITC",
    "LT", "AXISBANK", "ASIANPAINT", "MARUTI", "HCLTECH", "SUNPHARMA", "TITAN", "BAJFINANCE", "ULTRACEMCO", "WIPRO",
    "NESTLEIND", "NTPC", "POWERGRID", "M&M","JSWSTEEL", "TATASTEEL", "ADANIENT", "ADANIPORTS", "ONGC",
    "COALINDIA", "TECHM", "BAJAJFINSV", "GRASIM","DIVISLAB", "DRREDDY", "CIPLA", "BRITANNIA", "EICHERMOT",
    "APOLLOHOSP", "SBILIFE", "HDFCLIFE", "BPCL","TATACONSUM", "UPL", "HINDALCO", "BAJAJ-AUTO",
    "ADANIGREEN", "AMBUJACEM", "BANKBARODA","BOSCHLTD", "CHOLAFIN","DLF", "GAIL",
    "GODREJCP","JINDALSTEL","PIDILITIND", "PNB","SHREECEM", "SIEMENS", "TATAPOWER", "TORNTPHARM",
    "TRENT", "VEDL", "ZYDUSLIFE", "ABB","DMART","INDIGO",
    "IRFC", "JIOFIN", "LODHA", "MAXHEALTH", "PAYTM","VBL","BEL","ETERNAL",
    "SHRIRAMFIN","TMPV","ADANIENSOL","ADANIPOWER","BAJAJHLDNG","CGPOWER","CANBK","CUMMINSIND","HDFCAMC","HAL","HINDZINC",
    "HYUNDAI","INDHOTEL","MAZDOCK","MUTHOOTFIN","PFC","RECLTD","MOTHERSON","ENRIN","SOLARINDS","TVSMOTOR","TATACAP",
    "TMCV","UNIONBANK","UNITDSPR"
]

# Add .NS suffix for Yahoo Finance
nse_tickers = [f"{ticker}.NS" for ticker in NIFTY_100_TICKERS]

# Add Nifty 50 Index
INDEX_TICKER = "^NSEI"

print(f"Total Nifty 100 stocks to download: {len(nse_tickers)}")
print(f"Index ticker: {INDEX_TICKER}")
print(f"\nSample tickers: {nse_tickers[:5]}")

Total Nifty 100 stocks to download: 100
Index ticker: ^NSEI

Sample tickers: ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS']


## 3. Download Stock Data Function

In [52]:
def download_stock_data(ticker, start_date, end_date, max_retries=3):
    """
    Download OHLCV data for a single ticker from Yahoo Finance.
    
    Args:
        ticker: Stock ticker symbol (with .NS suffix for NSE stocks)
        start_date: Start date string (YYYY-MM-DD)
        end_date: End date string (YYYY-MM-DD)
        max_retries: Number of retry attempts
    
    Returns:
        DataFrame with OHLCV data or None if download fails
    """
    for attempt in range(max_retries):
        try:
            # Download data
            stock = yf.Ticker(ticker)
            df = stock.history(start=start_date, end=end_date, auto_adjust=False)
            
            if df.empty:
                return None
            
            # Reset index to get Date as column
            df = df.reset_index()
            
            # Keep only required columns
            df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
            
            # Add ticker column
            df['Ticker'] = ticker.replace('.NS', '')  # Remove .NS suffix for cleaner data
            
            # Reorder columns
            df = df[['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']]
            
            return df
            
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)  # Wait before retry
            else:
                print(f"Failed to download {ticker}: {str(e)}")
                return None
    
    return None

## 4. Download All Stocks Data

In [53]:
def download_all_stocks(tickers, start_date, end_date):
    """
    Download data for all tickers and combine into single DataFrame.
    
    Args:
        tickers: List of ticker symbols
        start_date: Start date string
        end_date: End date string
    
    Returns:
        Combined DataFrame with all stock data
    """
    all_data = []
    successful = []
    failed = []
    
    print(f"Downloading data for {len(tickers)} stocks...")
    print(f"Date range: {start_date} to {end_date}")
    print("-" * 50)
    
    for ticker in tqdm(tickers, desc="Downloading"):
        df = download_stock_data(ticker, start_date, end_date)
        
        if df is not None and not df.empty:
            all_data.append(df)
            successful.append(ticker)
        else:
            failed.append(ticker)
        
        # Small delay to avoid rate limiting
        time.sleep(0.1)
    
    print("-" * 50)
    print(f"Successfully downloaded: {len(successful)} stocks")
    print(f"Failed to download: {len(failed)} stocks")
    
    if failed:
        print(f"\nFailed tickers: {failed[:20]}{'...' if len(failed) > 20 else ''}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df, successful, failed
    else:
        return pd.DataFrame(), successful, failed

In [54]:
# Download all Nifty 500 stocks
stock_df, successful_tickers, failed_tickers = download_all_stocks(
    nse_tickers, START_DATE, END_DATE
)

print(f"\nStock data shape: {stock_df.shape}")

Date range: 2023-01-01 to 2026-01-01
--------------------------------------------------


Downloading:   0%|          | 0/100 [00:00<?, ?it/s]

Downloading: 100%|██████████| 100/100 [00:21<00:00,  4.58it/s]

--------------------------------------------------
Successfully downloaded: 100 stocks
Failed to download: 0 stocks

Stock data shape: (70190, 7)


## 5. Download Nifty Index Data

In [55]:
# Download Nifty 50 Index data
print(f"Downloading Nifty Index ({INDEX_TICKER})...")

index_df = download_stock_data(INDEX_TICKER, START_DATE, END_DATE)

if index_df is not None:
    # Use 'NIFTY50' as ticker name
    index_df['Ticker'] = 'NIFTY50'
    print(f"Nifty Index data shape: {index_df.shape}")
    print(f"Date range: {index_df['Date'].min()} to {index_df['Date'].max()}")
else:
    print("Failed to download Nifty Index data")
    index_df = pd.DataFrame()

Nifty Index data shape: (740, 7)
Date range: 2023-01-02 00:00:00+05:30 to 2025-12-31 00:00:00+05:30


In [56]:
# Combine stock and index data
if not index_df.empty:
    combined_df = pd.concat([stock_df, index_df], ignore_index=True)
else:
    combined_df = stock_df.copy()

print(f"Combined data shape: {combined_df.shape}")

Combined data shape: (70930, 7)


## 6. Data Processing

In [57]:
def process_stock_data(df):
    """
    Process and clean the stock data.
    
    Processing rules:
    - Convert Date to datetime
    - Sort by Date and Ticker
    - Remove duplicate rows
    - Remove rows where Close is NaN
    - Keep only trading days
    
    Args:
        df: Raw stock DataFrame
    
    Returns:
        Processed DataFrame
    """
    print("Processing stock data...")
    print(f"Initial shape: {df.shape}")
    
    # Make a copy
    df = df.copy()
    
    # 1. Convert Date to datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Remove timezone info if present
    if df['Date'].dt.tz is not None:
        df['Date'] = df['Date'].dt.tz_localize(None)
    
    # 2. Sort by Date and Ticker
    df = df.sort_values(['Date', 'Ticker']).reset_index(drop=True)
    print(f"After sorting: {df.shape}")
    
    # 3. Remove duplicate rows
    initial_rows = len(df)
    df = df.drop_duplicates(subset=['Date', 'Ticker'], keep='first')
    duplicates_removed = initial_rows - len(df)
    print(f"Duplicates removed: {duplicates_removed}")
    
    # 4. Remove rows where Close is NaN
    initial_rows = len(df)
    df = df.dropna(subset=['Close'])
    nan_removed = initial_rows - len(df)
    print(f"Rows with NaN Close removed: {nan_removed}")
    
    # 5. Ensure numeric types
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 6. Final sort
    df = df.sort_values(['Date', 'Ticker']).reset_index(drop=True)
    
    print(f"Final shape: {df.shape}")
    
    return df

In [58]:
# Process the combined data
processed_df = process_stock_data(combined_df)

Processing stock data...
Initial shape: (70930, 7)
After sorting: (70930, 7)
Duplicates removed: 0
Rows with NaN Close removed: 0
Final shape: (70930, 7)


## 7. Validation Checks

In [59]:
def validate_stock_data(df):
    """
    Perform validation checks on the processed stock data.
    
    Args:
        df: Processed stock DataFrame
    """
    print("=" * 60)
    print("VALIDATION REPORT")
    print("=" * 60)
    
    # 1. Dataset shape
    print(f"\n1. Dataset Shape: {df.shape}")
    print(f"   - Total rows: {len(df):,}")
    print(f"   - Total columns: {len(df.columns)}")
    
    # 2. Unique tickers
    unique_tickers = df['Ticker'].nunique()
    print(f"\n2. Unique Tickers: {unique_tickers}")
    print(f"   - Sample tickers: {df['Ticker'].unique()[:10].tolist()}")
    
    # 3. Date range
    print(f"\n3. Date Range:")
    print(f"   - Start Date: {df['Date'].min()}")
    print(f"   - End Date: {df['Date'].max()}")
    print(f"   - Total trading days: {df['Date'].nunique()}")
    
    # 4. Missing values
    print(f"\n4. Missing Values:")
    missing = df.isnull().sum()
    for col in df.columns:
        pct = (missing[col] / len(df)) * 100
        print(f"   - {col}: {missing[col]:,} ({pct:.2f}%)")
    
    # 5. Data types
    print(f"\n5. Data Types:")
    for col in df.columns:
        print(f"   - {col}: {df[col].dtype}")
    
    # 6. Value ranges
    print(f"\n6. Value Ranges:")
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        print(f"   - {col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")
    
    # 7. Sample rows
    print(f"\n7. Sample Rows:")
    print(df.head(10).to_string())
    
    print("\n" + "=" * 60)

In [60]:
# Run validation
validate_stock_data(processed_df)

VALIDATION REPORT

1. Dataset Shape: (70930, 7)
   - Total rows: 70,930
   - Total columns: 7

2. Unique Tickers: 101
   - Sample tickers: ['ABB', 'ADANIENT', 'ADANIGREEN', 'ADANIPORTS', 'ADANIPOWER', 'AMBUJACEM', 'APOLLOHOSP', 'ASIANPAINT', 'AXISBANK', 'BAJAJ-AUTO']

3. Date Range:
   - Start Date: 2023-01-02 00:00:00
   - End Date: 2025-12-31 00:00:00
   - Total trading days: 740

4. Missing Values:
   - Date: 0 (0.00%)
   - Ticker: 0 (0.00%)
   - Open: 0 (0.00%)
   - High: 0 (0.00%)
   - Low: 0 (0.00%)
   - Close: 0 (0.00%)
   - Volume: 0 (0.00%)

5. Data Types:
   - Date: datetime64[ns]
   - Ticker: object
   - Open: float64
   - High: float64
   - Low: float64
   - Close: float64
   - Volume: int64

6. Value Ranges:
   - Open: min=25.50, max=41700.00, mean=2642.87
   - High: min=26.00, max=41945.00, mean=2670.62
   - Low: min=25.40, max=41095.00, mean=2612.08
   - Close: min=25.50, max=41495.00, mean=2640.79
   - Volume: min=0.00, max=694984567.00, mean=7952722.37

7. Sample Rows:

## 8. Per-Ticker Statistics

In [61]:
# Calculate per-ticker statistics
ticker_stats = processed_df.groupby('Ticker').agg({
    'Date': ['min', 'max', 'count'],
    'Close': ['mean', 'std', 'min', 'max'],
    'Volume': 'mean'
}).round(2)

ticker_stats.columns = ['Start_Date', 'End_Date', 'Trading_Days', 
                        'Avg_Close', 'Std_Close', 'Min_Close', 'Max_Close',
                        'Avg_Volume']

print(f"Per-Ticker Statistics (showing first 20):")
print(ticker_stats.head(20).to_string())

Per-Ticker Statistics (showing first 20):
           Start_Date   End_Date  Trading_Days  Avg_Close  Std_Close  Min_Close  Max_Close   Avg_Volume
Ticker                                                                                                 
ABB        2023-01-02 2025-12-31           740    5484.15    1534.79    2679.95    9020.00    346485.80
ADANIENSOL 2023-08-16 2025-12-31           583     922.78     125.50     600.75    1275.20   2671286.85
ADANIENT   2023-01-02 2025-12-31           740    2591.11     449.81    1193.50    3855.30   3091700.21
ADANIGREEN 2023-01-02 2025-12-31           740    1249.08     410.76     462.20    2166.15   2869905.95
ADANIPORTS 2023-01-02 2025-12-31           740    1147.69     301.90     462.45    1590.15   5369656.72
ADANIPOWER 2023-01-02 2025-12-31           740     101.17      35.22      27.87     174.90  40195499.46
AMBUJACEM  2023-01-02 2025-12-31           740     527.66      84.05     329.90     695.00   4567515.13
APOLLOHOSP 2023-01-02 

In [62]:
# Check for tickers with low data coverage
low_coverage = ticker_stats[ticker_stats['Trading_Days'] < 100]
print(f"\nTickers with less than 100 trading days: {len(low_coverage)}")
if len(low_coverage) > 0:
    print(low_coverage.to_string())


Tickers with less than 100 trading days: 3
        Start_Date   End_Date  Trading_Days  Avg_Close  Std_Close  Min_Close  Max_Close   Avg_Volume
Ticker                                                                                              
TATACAP 2025-10-13 2025-12-31            55     326.94       5.86      316.9      344.0   6595946.20
TMCV    2025-11-12 2025-12-31            35     363.75      34.52      317.6      426.1  16976403.97
TMPV    2025-10-15 2025-12-31            53     375.41      25.01      343.4      417.0  12585411.49


## 9. Save to CSV

In [63]:
# Save processed data to CSV
processed_df.to_csv(OUTPUT_FILE, index=False)

# Verify saved file
file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)  # MB

print(f"\nData saved to: {OUTPUT_FILE}")
print(f"File size: {file_size:.2f} MB")


Data saved to: ../Market_Data/raw\raw_stock_data.csv
File size: 5.68 MB


In [64]:
# Verify by reading back
verification_df = pd.read_csv(OUTPUT_FILE, parse_dates=['Date'])

print(f"\nVerification - Read back data:")
print(f"Shape: {verification_df.shape}")
print(f"Columns: {verification_df.columns.tolist()}")
print(f"Date range: {verification_df['Date'].min()} to {verification_df['Date'].max()}")


Verification - Read back data:
Shape: (70930, 7)
Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
Date range: 2023-01-02 00:00:00 to 2025-12-31 00:00:00


## 10. Summary Report

In [65]:
print("\n" + "=" * 60)
print("STOCK DATA INGESTION - FINAL SUMMARY")
print("=" * 60)

print(f"\n📊 DATA COLLECTION SUMMARY:")
print(f"   - Total stocks attempted: {len(nse_tickers)}")
print(f"   - Successfully downloaded: {len(successful_tickers)}")
print(f"   - Failed to download: {len(failed_tickers)}")
print(f"   - Nifty Index included: {'Yes' if 'NIFTY50' in processed_df['Ticker'].values else 'No'}")

print(f"\n📁 DATASET STATISTICS:")
print(f"   - Total rows: {len(processed_df):,}")
print(f"   - Unique tickers: {processed_df['Ticker'].nunique()}")
print(f"   - Date range: {processed_df['Date'].min().date()} to {processed_df['Date'].max().date()}")
print(f"   - Trading days: {processed_df['Date'].nunique()}")

print(f"\n📋 COLUMN STRUCTURE:")
print(f"   - Columns: {processed_df.columns.tolist()}")

print(f"\n❓ MISSING VALUES:")
for col in processed_df.columns:
    missing = processed_df[col].isnull().sum()
    if missing > 0:
        print(f"   - {col}: {missing:,} missing")
    else:
        print(f"   - {col}: No missing values")

print(f"\n💾 OUTPUT:")
print(f"   - File: {OUTPUT_FILE}")
print(f"   - Size: {file_size:.2f} MB")

print("\n" + "=" * 60)
print("✅ Stock data ingestion completed successfully!")
print("=" * 60)


STOCK DATA INGESTION - FINAL SUMMARY

📊 DATA COLLECTION SUMMARY:
   - Total stocks attempted: 100
   - Successfully downloaded: 100
   - Failed to download: 0
   - Nifty Index included: Yes

📁 DATASET STATISTICS:
   - Total rows: 70,930
   - Unique tickers: 101
   - Date range: 2023-01-02 to 2025-12-31
   - Trading days: 740

📋 COLUMN STRUCTURE:
   - Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']

❓ MISSING VALUES:
   - Date: No missing values
   - Ticker: No missing values
   - Open: No missing values
   - High: No missing values
   - Low: No missing values
   - Close: No missing values
   - Volume: No missing values

💾 OUTPUT:
   - File: ../Market_Data/raw\raw_stock_data.csv
   - Size: 5.68 MB

✅ Stock data ingestion completed successfully!


In [66]:
processed_df["Ticker"].nunique()

101

In [67]:
processed_df["Date"].min()
processed_df["Date"].max()

Timestamp('2025-12-31 00:00:00')

In [68]:
processed_df.isnull().sum()

Date      0
Ticker    0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

In [69]:
processed_df.groupby("Ticker").size()

Ticker
ABB           740
ADANIENSOL    583
ADANIENT      740
ADANIGREEN    740
ADANIPORTS    740
             ... 
UPL           740
VBL           740
VEDL          740
WIPRO         740
ZYDUSLIFE     740
Length: 101, dtype: int64

In [70]:
# Output paths
RAW_DATA_DIR = "../Market_Data/raw"
PARQUET_FILE = os.path.join(RAW_DATA_DIR, "stock_data.parquet")

# Create directory if not exists
os.makedirs(RAW_DATA_DIR, exist_ok=True)
processed_df.to_parquet(PARQUET_FILE)

---

## Next Steps

1. **Notebook 03**: Calculate technical indicators using TA-Lib
2. **Notebook 04**: Download global market data
3. **Notebook 05**: Fetch news sentiment from Alpha Vantage

---

**Next Notebook:** `03_technical_indicators.ipynb`